# Experiment Lab

## Capstone challenge: combine the learning levers

This notebook is the final sandbox for the solar PV mini-project. The earlier notebooks separated the data, hypothesis, and optimizer/objective spaces so each diagnostic was easier to reason about. Here you can change those levers together.

The aim is to keep the same investigation structure: state a hypothesis, choose an intervention that tests it, inspect validation evidence, and only reveal the final test result after one configuration is locked.


In [ ]:
# Environment setup. The notebook is designed to run locally and in Colab.
import importlib.util
import os
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "nextgen2026-matplotlib")
)

REPO_URL = "https://github.com/nextgenerationgraduatesprogram/nextgen2026-mlai-workshops.git"
REPO_BRANCH = "workshop3"
PACKAGE_NAME = "nextgen2026_mlai_workshops"

if "google.colab" in sys.modules:
    repo_dir = Path("/content/nextgen2026-mlai-workshops")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPO_BRANCH,
                REPO_URL,
                str(repo_dir),
            ],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], check=True)
    missing_packages = [
        package_name
        for package_name, module_name in (("pandas", "pandas"), ("torch", "torch"))
        if importlib.util.find_spec(module_name) is None
    ]
    if missing_packages:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
    sys.path.insert(0, str(repo_dir / "src"))
else:
    repo_dir = None
    for possible_root in (Path.cwd(), Path.cwd().parent):
        possible_src = possible_root / "src"
        if (possible_src / PACKAGE_NAME).exists():
            repo_dir = possible_root
            sys.path.insert(0, str(possible_src))
            break

for module_name in list(sys.modules):
    if module_name == PACKAGE_NAME or module_name.startswith(f"{PACKAGE_NAME}."):
        del sys.modules[module_name]

print(f"Workshop 3 environment ready. Repository: {repo_dir or 'installed environment'}")


## 1. Challenge Contract

The validation set is the evidence used for tuning. The test set is held back for one final check after your group has fixed a final configuration.

The challenge criterion is stricter than the earlier workshop criterion, so a single small improvement is unlikely to be enough. You should expect to combine levers, but each combination should still have a reason.


In [ ]:
from nextgen2026_mlai_workshops import solar_pv as pv
import matplotlib.pyplot as plt
import numpy as np

show_advanced = False
challenge_seed = 7
challenge_bundle = pv.make_challenge_bundle(seed=challenge_seed)
criterion = challenge_bundle['criterion']

pv.print_table(
    ['Check', 'Acceptable if', 'Scope'],
    [
        ['Overall validation RMSE', f'RMSE <= {criterion.overall_rmse:.3f}', 'all validation examples'],
        [
            'Key-range validation RMSE',
            f'RMSE <= {criterion.key_range_rmse:.3f}',
            (
                f'irradiance >= {criterion.key_irradiance_min:.0f}; '
                f'ambient_temperature >= {criterion.key_ambient_min:.0f}; '
                f'{criterion.key_tilt_min:.0f} <= tilt_angle <= {criterion.key_tilt_max:.0f}'
            ),
        ],
    ],
)

pv.print_table(
    ['Split', 'Rows', 'Key-range rows'],
    [
        [split, len(pv.target_vector(challenge_bundle[split])), int(np.sum(pv.key_range_mask(challenge_bundle[split], criterion)))]
        for split in ('train', 'validation')
    ],
)


## 2. Baseline Under The Stricter Criterion

Start by running the fixed baseline against the challenge criterion. This gives your group a reference point for deciding whether later changes are meaningful.


In [ ]:
challenge_headers = [
    'Run',
    'Data policy',
    'Added rows',
    'Input dim',
    'Width',
    'Depth',
    'Activation',
    'Optimizer',
    'Loss',
    'Validation RMSE',
    'Key-range RMSE',
    'Criterion pass',
    'Best epoch',
]

baseline_result = pv.run_challenge_experiment(
    challenge_bundle,
    pv.challenge_config(),
    include_test=False,
    name='challenge baseline',
    show_advanced=show_advanced,
)
pv.print_table(challenge_headers, [baseline_result['row']])
pv.print_report(baseline_result['report'])
_ = pv.plot_training_curves(baseline_result['run'])
_ = pv.plot_observed_vs_predicted(baseline_result['run'], challenge_bundle, split='validation')


## 3. Full Option Surface

The table below lists the dials available in this lab. Change a small number at a time unless your hypothesis is explicitly about an interaction between levers.


In [ ]:
pv.print_table(['Space', 'Config name', 'Options'], pv.challenge_option_rows())


## 4. Choose Data Points To Collect Around

A data intervention can be a broad collection policy, or it can target specific examples where the current evidence is weak. The table below ranks validation rows using nearest-training distance and baseline residual. Use the row IDs as candidate anchors for `target_point_indices` in the control panel.


In [ ]:
point_candidate_rows = pv.challenge_candidate_point_rows(
    challenge_bundle,
    run=baseline_result['run'],
    top_k=10,
)
starter_target_points = [row[0] for row in point_candidate_rows[:3]]
pv.print_table(
    ['Validation row', 'Nearest train distance', 'Abs residual', 'Irradiance', 'Ambient temp', 'Tilt', 'Target'],
    point_candidate_rows,
)
print('Starter point IDs for a targeted data intervention:', starter_target_points)


## 5. Experiment Control Panel

Edit the values in this cell to define one experiment. Keep the hypothesis specific enough that your validation result can support or weaken it.


In [ ]:
experiment_name = 'trial 1'
experiment_hypothesis = 'The default visible-input baseline is the control run for later comparisons.'

# Data-space lever. Use 'none' with added_n = 0 for the baseline data.
# Use data_policy = 'target_selected_points' with target_point_indices to collect near specific examples.
data_policy = 'none'
added_n = 0
target_split = 'validation'
target_point_indices = []  # e.g. starter_target_points or [12, 45, 98]

# Hypothesis-space and optimizer/objective levers.
experiment_config = pv.challenge_config(
    input_columns=list(pv.VISIBLE_INPUTS),
    hidden_width=16,
    depth=2,
    activation='relu',
    regularity_strength='none',
    scaling='standard',
    initialization='he',
    optimizer='sgd',
    momentum=0.0,
    learning_rate=0.035,
    batch_size=32,
    epochs=220,
    loss='mse',
    huber_delta=0.08,
    selection_metric='rmse',
    seed=7,
)

print('Experiment:', experiment_name)
print('Hypothesis:', experiment_hypothesis)
print('Data policy:', data_policy, 'added rows:', added_n)
print('Target split:', target_split, 'target point indices:', target_point_indices)
print('Config:', experiment_config)


## 6. Run And Evaluate On Validation Evidence

This cell trains the configured model and prints validation evidence only. Use the result to decide what to try next or whether the final configuration is ready to lock.


In [ ]:
experiment_bundle = pv.make_challenge_bundle(
    seed=challenge_seed,
    data_policy=data_policy,
    added_n=added_n,
    target_split=target_split,
    target_point_indices=target_point_indices if data_policy == 'target_selected_points' else None,
)
experiment_result = pv.run_challenge_experiment(
    experiment_bundle,
    experiment_config,
    include_test=False,
    name=experiment_name,
    show_advanced=show_advanced,
)

pv.print_table(challenge_headers, [baseline_result['row'], experiment_result['row']])
print('\nHypothesis:', experiment_hypothesis)
pv.print_report(experiment_result['report'])
if experiment_result['run']['history']:
    _ = pv.plot_training_curves(experiment_result['run'])
    _ = pv.plot_observed_vs_predicted(experiment_result['run'], experiment_bundle, split='validation')


### Result Log

| Experiment | Hypothesis | Changed levers | Validation result | Conclusion |
|---|---|---|---|---|
| | | | | |


## 7. Interaction Probe

Use this section to compare whether a combination behaves differently from its parts. The starter probe compares data-only, measurement-only, and data-plus-measurement changes without revealing the final calibrated answer.


In [ ]:
interaction_probe_specs = [
    (
        'data only',
        'deployment_matched_mix',
        300,
        pv.challenge_config(epochs=160),
        None,
    ),
    (
        'selected diagnostic points',
        'target_selected_points',
        120,
        pv.challenge_config(epochs=160),
        starter_target_points,
    ),
    (
        'measurements only',
        'none',
        0,
        pv.challenge_config(input_columns=[*pv.VISIBLE_INPUTS, 'cloud_cover', 'panel_temperature'], epochs=160),
        None,
    ),
    (
        'data + measurements',
        'deployment_matched_mix',
        300,
        pv.challenge_config(input_columns=[*pv.VISIBLE_INPUTS, 'cloud_cover', 'panel_temperature'], epochs=160),
        None,
    ),
]

interaction_rows = []
interaction_results = {}
for probe_name, probe_policy, probe_added_n, probe_config, probe_points in interaction_probe_specs:
    probe_bundle = pv.make_challenge_bundle(
        seed=challenge_seed,
        data_policy=probe_policy,
        added_n=probe_added_n,
        target_point_indices=probe_points,
    )
    probe_result = pv.run_challenge_experiment(
        probe_bundle,
        probe_config,
        include_test=False,
        name=probe_name,
        show_advanced=show_advanced,
    )
    interaction_rows.append(probe_result['row'])
    interaction_results[probe_name] = probe_result

pv.print_table(challenge_headers, interaction_rows)


## 8. Final Locked Test Check

Set `selected_final_config` only after your group has chosen one final configuration from validation evidence. After this cell reveals test performance, do not use that result to continue tuning.


In [ ]:
selected_final_name = 'locked final run'
selected_final_data_policy = None
selected_final_added_n = 0
selected_final_target_split = 'validation'
selected_final_target_point_indices = []
selected_final_config = None

if selected_final_config is None or selected_final_data_policy is None:
    print('Set selected_final_data_policy, selected_final_added_n, and selected_final_config after locking one final choice.')
else:
    final_bundle = pv.make_challenge_bundle(
        seed=challenge_seed,
        data_policy=selected_final_data_policy,
        added_n=selected_final_added_n,
        target_split=selected_final_target_split,
        target_point_indices=(
            selected_final_target_point_indices
            if selected_final_data_policy == 'target_selected_points'
            else None
        ),
    )
    final_result = pv.run_challenge_experiment(
        final_bundle,
        selected_final_config,
        include_test=True,
        name=selected_final_name,
        show_advanced=show_advanced,
    )
    final_headers = [*challenge_headers, 'Test RMSE']
    pv.print_table(final_headers, [final_result['row']])
    pv.print_report(final_result['report'])


## 9. Report-Back Template

| Question | Your answer |
|---|---|
| What failure hypothesis did you test? | |
| Which evidence motivated that hypothesis? | |
| Which levers did you change? | |
| What happened on validation evidence? | |
| Did the final locked model meet the challenge criterion? | |
| What would you test next if this were a real project? | |
